In [88]:
import pandas as pd
from sklearn.metrics import jaccard_score
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Step 1: Load Data
user_data = pd.read_csv('UserData.csv')
user_historical_view = pd.read_csv('UserHistoricalView.csv')
book_data = pd.read_csv('bookData.csv',encoding='ISO-8859-1')


In [89]:
print(user_data.head())

   userid                        location   age
0   11676                   n/a, n/a, n/a   NaN
1   16795   mechanicsville, maryland, usa  47.0
2   22625         lynbrook, new york, usa  54.0
3   35859          duluth, minnesota, usa   NaN
4   95359  charleston, west virginia, usa  33.0


In [90]:
print(user_historical_view)

    userid        isbn
0    11676    60938455
1    11676   316096199
2    11676   316569321
3    11676   312195516
4    11676   345361792
..     ...         ...
84  271448  043935806X
85  271448  044023722X
86  271448   446672211
87  271448   452282829
88  271448   743237188

[89 rows x 2 columns]


In [91]:
print(book_data)

           isbn                                          booktitle  \
0     440234743                                      The Testament   
1     971880107                                        Wild Animus   
2     345417623                                           Timeline   
3     446310786                              To Kill a Mockingbird   
4     671027360                                  Angels and Demons   
..          ...                                                ...   
109   439064864   Harry Potter and the Chamber of Secrets (Book 2)   
110  043935806X  Harry Potter and the Order of the Phoenix (Boo...   
111   440220602                                        The Chamber   
112   671001795                                  Two for the Dough   
113   440222656                                The Horse Whisperer   

           bookauthor  yearofpublication                   publisher  \
0        John Grisham               1999                        Dell   
1        Rich S

In [92]:
# Step 2: Create User Profiles
merged_data = pd.merge(user_data, user_historical_view, on='userid')
print(merged_data)

    userid                 location   age        isbn
0    11676            n/a, n/a, n/a   NaN    60938455
1    11676            n/a, n/a, n/a   NaN   316096199
2    11676            n/a, n/a, n/a   NaN   316569321
3    11676            n/a, n/a, n/a   NaN   312195516
4    11676            n/a, n/a, n/a   NaN   345361792
..     ...                      ...   ...         ...
84  271448  ona, west virginia, usa  27.0  043935806X
85  271448  ona, west virginia, usa  27.0  044023722X
86  271448  ona, west virginia, usa  27.0   446672211
87  271448  ona, west virginia, usa  27.0   452282829
88  271448  ona, west virginia, usa  27.0   743237188

[89 rows x 4 columns]


In [93]:
user_book_data = pd.merge(merged_data, book_data[['isbn', 'Synopsis']], on='isbn')
print(user_book_data)

    userid                      location   age        isbn  \
0    11676                 n/a, n/a, n/a   NaN    60938455   
1    11676                 n/a, n/a, n/a   NaN   316096199   
2    11676                 n/a, n/a, n/a   NaN   316569321   
3    11676                 n/a, n/a, n/a   NaN   312195516   
4    35859        duluth, minnesota, usa   NaN   312195516   
..     ...                           ...   ...         ...   
84  204864  simi valley, california, usa  47.0   812550706   
85  271448       ona, west virginia, usa  27.0    60928336   
86  271448       ona, west virginia, usa  27.0   312291639   
87  271448       ona, west virginia, usa  27.0  044023722X   
88  271448       ona, west virginia, usa  27.0   743237188   

                                             Synopsis  
0   Fast Food Nation: The Dark Side of the All-Ame...  
1   Lucky describes her experience of being raped ...  
2   Powerfully acted adaptation of Janet Fitch's b...  
3   Dinah opens the story by re

In [94]:
user_profiles = user_book_data.groupby('userid')['Synopsis'].apply(lambda x: ' '.join(map(str, x))).reset_index()
group_no = 10
user_profiles.to_csv(f'Part2_File1_Profile_Group[group_10].csv', index=False)


In [95]:
# Step 3: Vectorize the Book Synopses for Jaccard Similarity
vectorizer = CountVectorizer(binary=True, stop_words='english')
book_vectors = vectorizer.fit_transform(book_data['Synopsis'].fillna(''))


In [96]:
# Step 4: Calculate Jaccard Similarity Matrix
jaccard_sim_matrix = np.zeros((book_vectors.shape[0], book_vectors.shape[0]))

for i in range(book_vectors.shape[0]):
    for j in range(book_vectors.shape[0]):
        jaccard_sim_matrix[i, j] = jaccard_score(book_vectors.toarray()[i], book_vectors.toarray()[j])


/Users/waiphyooo/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 due to no true or predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [97]:
# Convert to DataFrame for better handling
sim_matrix_df = pd.DataFrame(jaccard_sim_matrix, index=book_data['isbn'], columns=book_data['isbn'])


In [98]:
# Store the similarity matrix
sim_matrix_df.to_csv(f'Part2_File2_Model_Group[group_10].csv')

In [99]:
# Step 5: Recommend Top-10 Not-Yet-Read Books
recommendations = []
for user in user_data['userid']:
    read_books = user_historical_view[user_historical_view['userid'] == user]['isbn'].tolist()
    unread_books = [isbn for isbn in book_data['isbn'] if isbn not in read_books]

    user_recommendations = []
    for unread_book in unread_books:
        similarity_scores = sim_matrix_df.loc[unread_book, read_books]
        avg_similarity = similarity_scores.mean() if not similarity_scores.empty else 0
        user_recommendations.append((unread_book, avg_similarity))

    # Sort by similarity and pick the top 10
    top_10_books = sorted(user_recommendations, key=lambda x: x[1], reverse=True)[:10]
    
    for isbn, similarity in top_10_books:
        book_title = book_data.loc[book_data['isbn'] == isbn, 'booktitle'].values[0]
        recommendations.append({
            'User ID': user,
            'Book’s ISBN': isbn,
            'Book’s Title': book_title,
            'Jaccard Similarity': round(similarity, 3)
        })

In [100]:
# Store recommendations
recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv(f'Part2_File3_Recommendation_Group[group_10].csv', index=False)


In [101]:
# Display recommendations with spacing
print("Top 10 Recommendations for Each User:\n")
last_user_id = None
for _, row in recommendations_df.iterrows():
    if last_user_id and row['User ID'] != last_user_id:
        print()  # Add a blank line for separation
    print(f"User ID: {row['User ID']}")
    print(f"  Book’s ISBN: {row['Book’s ISBN']}")
    print(f"  Book’s Title: {row['Book’s Title']}")
    print(f"  Jaccard Similarity: {row['Jaccard Similarity']:.3f}")
    last_user_id = row['User ID']

Top 10 Recommendations for Each User:

User ID: 11676
  Book’s ISBN: 590353403
  Book’s Title: Harry Potter and the Sorcerer s Stone (Book 1)
  Jaccard Similarity: 0.078
User ID: 11676
  Book’s ISBN: 439064872
  Book’s Title: Harry Potter and the Chamber of Secrets (Book 2)
  Jaccard Similarity: 0.030
User ID: 11676
  Book’s ISBN: 804114986
  Book’s Title: The Bonesetter s Daughter
  Jaccard Similarity: 0.026
User ID: 11676
  Book’s ISBN: 439136369
  Book’s Title: Harry Potter and the Prisoner of Azkaban (Book 3)
  Jaccard Similarity: 0.024
User ID: 11676
  Book’s ISBN: 60987103
  Book’s Title: Wicked: The Life and Times of the Wicked Witch of the West
  Jaccard Similarity: 0.024
User ID: 11676
  Book’s ISBN: 684874350
  Book’s Title: ANGELA S ASHES
  Jaccard Similarity: 0.023
User ID: 11676
  Book’s ISBN: 671003755
  Book’s Title: She s Come Undone 
  Jaccard Similarity: 0.021
User ID: 11676
  Book’s ISBN: 316776963
  Book’s Title: Me Talk Pretty One Day
  Jaccard Similarity: 0.020
Us

In [102]:
import pandas as pd
 
# Load data
recommendations_df = pd.read_csv('Part2_File3_Recommendation_Groupgroup_10.csv')
test_user_answers = pd.read_csv('TestUserAnswers.csv')
 


In [103]:
recommendations_df.head()

,User ID,Book’s ISBN,Book’s Title,Jaccard Similarity
0,11676,590353403,Harry Potter and the Sorcerer s Stone (Book 1),0.078
1,11676,439064872,Harry Potter and the Chamber of Secrets (Book 2),0.030
2,11676,804114986,The Bonesetter s Daughter,0.026
3,11676,439136369,Harry Potter and the Prisoner of Azkaban (Book 3),0.024
4,11676,60987103,Wicked: The Life and Times of the Wicked Witch...,0.024


In [104]:
test_user_answers.head()

,userid,isbn
0,11676,60502258
1,11676,60987103
2,11676,61009059
3,11676,014028009X
4,11676,142000205


In [105]:
# Calculate Precision, Recall, and F-measure
evaluation_results = []
 
for user_id in test_user_answers['userid'].unique():
    # Get recommended books for the user
    recommended_books = recommendations_df[recommendations_df['User ID'] == user_id]['Book’s ISBN'].tolist()
    # Get relevant books from the test dataset
    relevant_books = test_user_answers[test_user_answers['userid'] == user_id]['isbn'].tolist()
    # Calculate 'a' - number of relevant books in the top 10 recommendations
    a = len(set(recommended_books) & set(relevant_books))
    # Total relevant items for the user in the test dataset
    total_relevant_items = len(relevant_books)
    # Precision, Recall, and F-measure
    precision = a / 90
    recall = a / total_relevant_items if total_relevant_items > 0 else 0
    f_measure = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    # Append results
    evaluation_results.append({
        'User ID': user_id,
        'Precision': precision,
        'Recall': recall,
        'F-measure': f_measure
    })
 


In [107]:
# Convert results to DataFrame
evaluation_df = pd.DataFrame(evaluation_results)
print(evaluation_df)
# Save the evaluation metrics to a CSV file
evaluation_df.to_csv('Part2_File4_Evaluation_Group[group_10].csv', index=False)

   User ID  Precision    Recall  F-measure
0    11676   0.088889  0.170213   0.116788
1    16795   0.022222  0.074074   0.034188
2    22625   0.033333  0.428571   0.061856
3    35859   0.011111  0.076923   0.019417
4    95359   0.022222  0.090909   0.035714
5   104636   0.011111  0.066667   0.019048
6   110912   0.022222  0.222222   0.040404
7   204864   0.011111  0.100000   0.020000
8   271448   0.022222  0.181818   0.039604
